# Recorte do rosto da onça com fundo transparente

Este notebook:
1. Carrega a imagem original (`mascot/konnen-mascot-black-on-white.png`)
2. Recorta a imagem ao redor do rosto da onça (removendo o excesso de margem branca)
3. Deixa apenas o fundo externo transparente, preservando o branco interno do rosto (olhos, bochechas, orelhas)
4. Salva o resultado final

In [ ]:
from PIL import Image
import collections

INPUT_PATH = "../mascot/konnen-mascot-black-on-white.png"
CROPPED_PATH = "saida/konnen-mascot-black-on-white.png"
import os
os.makedirs("saida", exist_ok=True)
OUTPUT_PATH = "saida/konnen-mascot-black.png"

# fração de padding em torno do conteúdo detectado (ajuste para recortar mais ou menos)
PAD_FRACTION = 0.06

## 1. Detectar a área do desenho e recortar (crop)

In [ ]:
img = Image.open(INPUT_PATH).convert("RGBA")
w, h = img.size

# encontra o bounding box do conteúdo não-branco
gray = img.convert("L")
bbox = gray.point(lambda p: 255 if p < 250 else 0).getbbox()
left, top, right, bottom = bbox
bw, bh = right - left, bottom - top
print("bbox do conteúdo:", bbox)

# aplica padding
pad_x = int(bw * PAD_FRACTION)
pad_y = int(bh * PAD_FRACTION)

nl = max(0, left - pad_x)
nt = max(0, top - pad_y)
nr = min(w, right + pad_x)
nb = min(h, bottom + pad_y)

# torna o recorte quadrado, centralizado no conteúdo
side = max(nr - nl, nb - nt)
cx, cy = (nl + nr) // 2, (nt + nb) // 2
half = side // 2

cl = max(0, cx - half)
ct = max(0, cy - half)
cr = min(w, cl + side)
cb = min(h, ct + side)
cl = cr - side if cr - side >= 0 else 0
ct = cb - side if cb - side >= 0 else 0

cropped = img.crop((cl, ct, cr, cb))
cropped.save(CROPPED_PATH)
print("tamanho recortado:", cropped.size)
cropped

## 2. Tornar transparente apenas o fundo externo (flood fill a partir das bordas)

In [ ]:
def is_white(p, threshold=240):
    r, g, b, a = p
    return r > threshold and g > threshold and b > threshold


def make_outer_background_transparent(image):
    w, h = image.size
    px = image.load()

    visited = [[False] * w for _ in range(h)]
    mask = Image.new("L", (w, h), 0)  # 255 = tornar transparente
    mpx = mask.load()

    q = collections.deque()

    # semeia o flood fill a partir de todos os pixels brancos da borda
    for x in range(w):
        for y in (0, h - 1):
            if not visited[y][x] and is_white(px[x, y]):
                visited[y][x] = True
                q.append((x, y))
    for y in range(h):
        for x in (0, w - 1):
            if not visited[y][x] and is_white(px[x, y]):
                visited[y][x] = True
                q.append((x, y))

    while q:
        x, y = q.popleft()
        mpx[x, y] = 255
        for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
            if 0 <= nx < w and 0 <= ny < h and not visited[ny][nx] and is_white(px[nx, ny]):
                visited[ny][nx] = True
                q.append((nx, ny))

    datas = list(image.getdata())
    mdata = list(mask.getdata())
    new = [
        (r, g, b, 0) if m == 255 else (r, g, b, a)
        for (r, g, b, a), m in zip(datas, mdata)
    ]
    result = image.copy()
    result.putdata(new)
    return result


final_img = make_outer_background_transparent(cropped)
final_img.save(OUTPUT_PATH)
print("salvo em:", OUTPUT_PATH)
final_img

## 3. Conferir o resultado sobre um fundo colorido (só para visualização)

In [ ]:
preview_bg = Image.new("RGBA", final_img.size, (0, 200, 200, 255))
preview_bg.paste(final_img, (0, 0), final_img)
preview_bg.convert("RGB")